# Redes neuronales secuenciales GRU y TCN (GPU, Google Colab)

Este notebook entrena los modelos de comparación **GRU + contexto** y **TCN + contexto** (y una MLP de control) sobre ventanas crudas de 6 h, prediciendo la trayectoria completa ΔPO₄ de 2 a 60 min.

1. *Entorno de ejecución → Cambiar tipo → GPU (T4)*.
2. Ejecuta todas las celdas y sube `dataset.csv` cuando se solicite.
3. Descarga `resultados_colab.zip` al final y copia su contenido en `resultados/colab/` del repositorio.

Las variables de entrada y la partición son **idénticas** a las del repositorio (celda siguiente = copia de `src/po4_prediccion/datos.py`).

In [ ]:
import os, math, time, random
import numpy as np, pandas as pd, torch
from torch import nn
from google.colab import files
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', dev)
if not os.path.exists('dataset.csv'):
    files.upload()
os.makedirs('resultados_colab', exist_ok=True)

In [ ]:
%%writefile datos.py
"""Carga de datos, variables de entrada causales y partición cronológica.

Toda variable calculada en el instante de origen t usa únicamente
observaciones con tiempo <= t. El objetivo es T1_PO4(t + h), con h = 10, 30
y 60 min (5, 15 y 30 pasos de 2 min).
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path(__file__).resolve().parents[2]
DATOS = RAIZ / "data" / "dataset.csv"
RESULTADOS = RAIZ / "resultados"
TABLAS = RESULTADOS / "tablas"
PREDICCIONES = RESULTADOS / "predicciones"
MODELOS = RESULTADOS / "modelos"
COLAB = RESULTADOS / "colab"
FIGURAS = RAIZ / "figuras"

OBJETIVO = "T1_PO4"
HORIZONTES = {10: 5, 30: 15, 60: 30}  # minutos -> pasos de 2 min
CALENTAMIENTO = 400  # pasos iniciales descartados para que toda ventana esté definida
SEMILLA = 42


def asegurar_directorios() -> None:
    for d in (TABLAS, PREDICCIONES, MODELOS, COLAB, FIGURAS):
        d.mkdir(parents=True, exist_ok=True)


def cargar_datos() -> pd.DataFrame:
    """Lee el CSV SCADA de la PTAR de Agtrup y lo ordena cronológicamente."""
    df = pd.read_csv(DATOS, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    return df


# --------------------------------------------------------------------------
# Reloj de fase de la operación alternada
# --------------------------------------------------------------------------
def _pasos_desde_cambio(s: pd.Series) -> np.ndarray:
    """Número de pasos transcurridos desde el último cambio de valor de s."""
    grupo = s.ne(s.shift()).cumsum()
    return s.groupby(grupo).cumcount().to_numpy()


def _duracion_tramo_previo(s: pd.Series, k: int = 1) -> np.ndarray:
    """Duración del k-ésimo tramo *completo* anterior de s (causal)."""
    grupo = s.ne(s.shift()).cumsum().to_numpy()
    duraciones = pd.Series(grupo).value_counts().sort_index().to_numpy()
    previo = np.full(len(grupo), np.nan)
    idx = grupo - 1 - k
    ok = idx >= 0
    previo[ok] = duraciones[idx[ok]]
    return previo


def reloj_de_fase(d: pd.DataFrame) -> pd.DataFrame:
    """Variables que ubican el instante t dentro del ciclo de operación alternada.

    Un ciclo comienza cuando la fase de entrada pasa de 1 a 2 (inicio de la
    alimentación del tanque 1).
    """
    inl = d["PROCESSPHASE_INLET"].round().clip(1, 2).astype(int)
    out = d["PROCESSPHASE_OUTLET"].round().clip(1, 2).astype(int)
    f = pd.DataFrame(index=d.index)
    f["inl"] = inl
    f["out"] = out
    f["inl_out_state"] = (inl - 1) * 2 + (out - 1)
    f["since_inl_change"] = _pasos_desde_cambio(inl)
    f["since_out_change"] = _pasos_desde_cambio(out)
    f["prev_inl_run"] = _duracion_tramo_previo(inl, 1)
    f["prev_inl_run2"] = _duracion_tramo_previo(inl, 2)
    f["prev_out_run"] = _duracion_tramo_previo(out, 1)
    inicio = inl.eq(2) & inl.shift().eq(1)
    ciclo = inicio.cumsum()
    f["cycle_pos"] = d.groupby(ciclo).cumcount().to_numpy()
    # Duración del último ciclo completo, conocida en el instante t.
    ciclo_np = ciclo.to_numpy()
    tamanos = ciclo.value_counts().sort_index()
    ultimo = np.full(len(d), np.nan)
    previo = ciclo_np - 1
    m = previo >= tamanos.index.min()
    ultimo[m] = tamanos.reindex(previo[m]).to_numpy()
    f["last_cycle_len"] = ultimo
    f["cycle_frac"] = f["cycle_pos"] / f["last_cycle_len"]
    # Reloj de aireación: O2 > 0.3 mg/L se considera aireado.
    aire = (d["T1_O2"] > 0.3).astype(int)
    f["aerated"] = aire
    f["since_aer_change"] = _pasos_desde_cambio(aire)
    return f


# --------------------------------------------------------------------------
# Conjunto completo de 153 variables de entrada
# --------------------------------------------------------------------------
def construir_variables(d: pd.DataFrame) -> pd.DataFrame:
    po = d[OBJETIVO]
    f = reloj_de_fase(d)
    for c in ["IN_METAL_Q", "T1_O2", "METAL_Q", "TEMPERATURE", "IN_Q", "MAX_CF", "T1_NH4", "T1_PO4"]:
        f[c] = d[c]
    # Historia densa de PO4 (aprox. dos ciclos).
    for lag in list(range(1, 11)) + list(range(12, 31, 2)) + list(range(33, 91, 3)):
        f[f"po4_l{lag}"] = po.shift(lag)
    for lag in (1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 60):
        f[f"nh4_l{lag}"] = d["T1_NH4"].shift(lag)
        f[f"o2_l{lag}"] = d["T1_O2"].shift(lag)
    for lag in (1, 2, 5, 10, 20):
        f[f"inl_l{lag}"] = f["inl"].shift(lag)
        f[f"q_l{lag}"] = d["IN_Q"].shift(lag)
    # Estadísticos móviles (incluyen t; causales).
    for w in (5, 15, 40, 90, 360, 720):
        f[f"po4_mean_{w}"] = po.rolling(w).mean()
        f[f"po4_max_{w}"] = po.rolling(w).max()
        f[f"po4_min_{w}"] = po.rolling(w).min()
        f[f"nh4_mean_{w}"] = d["T1_NH4"].rolling(w).mean()
        f[f"o2_mean_{w}"] = d["T1_O2"].rolling(w).mean()
        f[f"q_mean_{w}"] = d["IN_Q"].rolling(w).mean()
        f[f"fe_sum_{w}"] = d["IN_METAL_Q"].rolling(w).sum()
        f[f"pac_sum_{w}"] = d["METAL_Q"].rolling(w).sum()
    f["po4_std_40"] = po.rolling(40).std()
    f["po4_d1"] = po.diff(1)
    f["po4_d3"] = po.diff(3)
    f["nh4_d3"] = d["T1_NH4"].diff(3)
    f["since_po4_change"] = _pasos_desde_cambio(po)
    # Valor en la misma posición del ciclo anterior (t - duración del último ciclo).
    L = f["last_cycle_len"].fillna(40).clip(5, 300).astype(int).to_numpy()
    idx = np.clip(np.arange(len(d)) - L, 0, None)
    f["po4_prev_cycle_same_pos"] = po.to_numpy()[idx]
    f["nh4_prev_cycle_same_pos"] = d["T1_NH4"].to_numpy()[idx]
    minuto = d["date"].dt.hour * 60 + d["date"].dt.minute
    dia = d["date"].dt.dayofyear
    f["tod_sin"] = np.sin(2 * np.pi * minuto / 1440)
    f["tod_cos"] = np.cos(2 * np.pi * minuto / 1440)
    f["dow"] = d["date"].dt.dayofweek
    f["year_sin"] = np.sin(2 * np.pi * dia / 365.25)
    f["year_cos"] = np.cos(2 * np.pi * dia / 365.25)
    return f


FAMILIAS = {
    "PO₄ actual": lambda c: c == "T1_PO4",
    "Historia de PO₄": lambda c: c.startswith("po4_") or c == "since_po4_change",
    "Fase y reloj del ciclo": lambda c: c in {"inl", "out", "inl_out_state", "since_inl_change", "since_out_change",
                                              "prev_inl_run", "prev_inl_run2", "prev_out_run", "cycle_pos",
                                              "last_cycle_len", "cycle_frac", "aerated", "since_aer_change"}
                                    or c.startswith("inl_l"),
    "NH₄ (actual e historia)": lambda c: c == "T1_NH4" or c.startswith("nh4_"),
    "O₂ disuelto": lambda c: c == "T1_O2" or c.startswith("o2_"),
    "Caudal y carga hidráulica": lambda c: c in {"IN_Q", "MAX_CF"} or c.startswith("q_"),
    "Dosificación química (Fe, PAC)": lambda c: c in {"IN_METAL_Q", "METAL_Q"} or c.startswith(("fe_", "pac_")),
    "Temperatura y calendario": lambda c: c in {"TEMPERATURE", "tod_sin", "tod_cos", "dow", "year_sin", "year_cos"},
}

ACTUALES = ["inl", "out", "inl_out_state", "IN_METAL_Q", "T1_O2", "METAL_Q", "TEMPERATURE", "IN_Q",
            "MAX_CF", "T1_NH4", "T1_PO4", "tod_sin", "tod_cos", "dow", "year_sin", "year_cos"]
RELOJ = ["since_inl_change", "since_out_change", "prev_inl_run", "prev_inl_run2", "prev_out_run",
         "cycle_pos", "last_cycle_len", "cycle_frac", "aerated", "since_aer_change"]


def familia_de(columna: str) -> str:
    hits = [k for k, fn in FAMILIAS.items() if fn(columna)]
    assert len(hits) == 1, (columna, hits)
    return hits[0]


def construir_dataset():
    """Devuelve (datos crudos, variables + objetivos) alineados por origen."""
    crudo = cargar_datos()
    f = construir_variables(crudo)
    for minutos, pasos in HORIZONTES.items():
        f[f"target_{minutos}"] = crudo[OBJETIVO].shift(-pasos)
    keep = slice(CALENTAMIENTO, len(crudo) - max(HORIZONTES.values()))
    return crudo.iloc[keep].reset_index(drop=True), f.iloc[keep].reset_index(drop=True)


def columnas_entrada(f: pd.DataFrame) -> list[str]:
    return [c for c in f.columns if not c.startswith("target_")]


def particion_cronologica(n: int, pasos: int):
    """70/15/15 cronológico; se purgan `pasos` orígenes antes de cada frontera
    para que ninguna etiqueta futura cruce al conjunto siguiente."""
    n0, n1 = int(n * .70), int(n * .85)
    return np.arange(0, n0 - pasos), np.arange(n0, n1 - pasos), np.arange(n1, n)


FOLDS_ESTACIONALES = [("2022-08-01", "2022-11-01"), ("2022-11-01", "2023-02-01"),
                      ("2023-02-01", "2023-05-01"), ("2023-05-01", "2023-08-01")]
DIAS_VALIDACION = 30


def particion_fold(fechas: pd.Series, inicio: str, fin: str, pasos: int):
    """Validación por origen móvil: entrenamiento = todo hasta 30 días antes
    del trimestre; validación = esos 30 días; prueba = el trimestre."""
    s, e = pd.Timestamp(inicio, tz="UTC"), pd.Timestamp(fin, tz="UTC")
    v = s - pd.Timedelta(days=DIAS_VALIDACION)
    d = fechas.to_numpy()
    n_v, n_s, n_e = np.searchsorted(d, v), np.searchsorted(d, s), np.searchsorted(d, e)
    return np.arange(0, n_v - pasos), np.arange(n_v, n_s - pasos), np.arange(n_s, n_e - pasos)

In [ ]:
from pathlib import Path
import datos
datos.DATOS = Path('dataset.csv')
from datos import CALENTAMIENTO, HORIZONTES, OBJETIVO, cargar_datos, columnas_entrada, construir_dataset, particion_cronologica
raw, f2 = construir_dataset()
cols2 = columnas_entrada(f2)
po = raw[OBJETIVO].to_numpy().astype('float32')
N = len(raw)
full = cargar_datos()[OBJETIVO].to_numpy().astype('float32')[CALENTAMIENTO:]
K = 30
TRAJ = np.stack([full[k:k + N] for k in range(1, K + 1)], 1) - po[:, None]
tr, va, te = particion_cronologica(N, K)
HORIZONS = HORIZONTES
print(N, len(cols2), TRAJ.shape, len(tr), len(va), len(te))

In [ ]:
# Sequence channels (per time step) and context vector (v2 features), train-standardized.
seq = np.column_stack([
    raw.T1_PO4, raw.T1_NH4, raw.T1_O2, raw.IN_Q / 1000, raw.IN_METAL_Q / 100, raw.METAL_Q / 10,
    raw.PROCESSPHASE_INLET, raw.PROCESSPHASE_OUTLET, raw.MAX_CF / 100, raw.TEMPERATURE / 10,
    f2.since_inl_change / 40, f2.cycle_frac.fillna(0).clip(0, 3),
]).astype('float32')
s_mu, s_sd = seq[tr].mean(0), seq[tr].std(0) + 1e-6
SEQ = torch.from_numpy((seq - s_mu) / s_sd).to(dev)
ctx = f2[cols2].to_numpy('float32')
c_mu, c_sd = np.nanmean(ctx[tr], 0), np.nanstd(ctx[tr], 0) + 1e-6
CTX = torch.from_numpy(np.nan_to_num((ctx - c_mu) / c_sd).astype('float32')).to(dev)
y_mu, y_sd = TRAJ[tr].mean(0), TRAJ[tr].std(0) + 1e-6
YT = torch.from_numpy(((TRAJ - y_mu) / y_sd).astype('float32')).to(dev)
W = 180
OFF = torch.arange(-W + 1, 1, device=dev)

def batch(idx):
    i = torch.as_tensor(idx, device=dev)
    win = SEQ[(i[:, None] + OFF).clamp(min=0)]  # (B, W, C), strictly past + present
    return win, CTX[i], YT[i]

In [ ]:
class MLPv2(nn.Module):
    def __init__(s, n_ctx, k=K, w=512):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(n_ctx, w), nn.LayerNorm(w), nn.GELU(), nn.Dropout(.1),
                              nn.Linear(w, w), nn.LayerNorm(w), nn.GELU(), nn.Dropout(.1),
                              nn.Linear(w, w), nn.LayerNorm(w), nn.GELU(), nn.Linear(w, k))
    def forward(s, win, c): return s.net(c)

class GRUCtx(nn.Module):
    def __init__(s, n_ch, n_ctx, k=K, h=128):
        super().__init__()
        s.gru = nn.GRU(n_ch, h, num_layers=2, batch_first=True, dropout=.1)
        s.ctx = nn.Sequential(nn.Linear(n_ctx, 256), nn.GELU(), nn.Dropout(.1))
        s.head = nn.Sequential(nn.Linear(h + 256, 256), nn.GELU(), nn.Linear(256, k))
    def forward(s, win, c):
        o, _ = s.gru(win)
        return s.head(torch.cat([o[:, -1], s.ctx(c)], 1))

class CausalConv(nn.Module):
    def __init__(s, ci, co, d):
        super().__init__(); s.pad = 2 * d; s.conv = nn.Conv1d(ci, co, 3, dilation=d)
    def forward(s, x): return s.conv(nn.functional.pad(x, (s.pad, 0)))

class TCNCtx(nn.Module):
    def __init__(s, n_ch, n_ctx, k=K, h=96):
        super().__init__()
        s.inp = nn.Conv1d(n_ch, h, 1)
        s.blocks = nn.ModuleList([CausalConv(h, h, d) for d in (1, 2, 4, 8, 16, 32, 64)])
        s.norms = nn.ModuleList([nn.GroupNorm(8, h) for _ in s.blocks])
        s.ctx = nn.Sequential(nn.Linear(n_ctx, 256), nn.GELU(), nn.Dropout(.1))
        s.head = nn.Sequential(nn.Linear(h + 256, 256), nn.GELU(), nn.Linear(256, k))
    def forward(s, win, c):
        x = s.inp(win.transpose(1, 2))
        for b, n in zip(s.blocks, s.norms):
            x = x + nn.functional.gelu(n(b(x)))
        return s.head(torch.cat([x[:, :, -1], s.ctx(c)], 1))

In [ ]:
def train(model, epochs=25, bs=2048, lr=2e-3, patience=5):
    model.to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    spe = math.ceil(len(tr) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * spe)
    scaler = torch.amp.GradScaler(enabled=dev.type == 'cuda')
    best, state, stale = 1e9, None, 0
    for ep in range(epochs):
        model.train(); perm = np.random.permutation(tr); t0 = time.time()
        for i in range(spe):
            win, c, y = batch(perm[i * bs:(i + 1) * bs])
            with torch.autocast(dev.type, enabled=dev.type == 'cuda'):
                loss = nn.functional.huber_loss(model(win, c).float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        vl = float(((predict(model, va, raw_scale=False) - YT[va].cpu().numpy()) ** 2).mean())
        print(f'ep {ep} val_mse {vl:.4f} ({time.time() - t0:.0f}s)')
        if vl < best - 1e-4: best, stale, state = vl, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience: break
    model.load_state_dict(state)
    return model

@torch.no_grad()
def predict(model, idx, raw_scale=True, bs=8192):
    model.eval(); out = []
    for i in range(0, len(idx), bs):
        win, c, _ = batch(idx[i:i + bs])
        out.append(model(win, c).float().cpu().numpy())
    out = np.concatenate(out)
    return out * y_sd + y_mu + po[idx, None] if raw_scale else out

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
def met(y, p, persist):
    p = np.clip(p, 0, 6)
    return dict(r2=r2_score(y, p), rmse=mean_squared_error(y, p) ** .5, mae=mean_absolute_error(y, p),
                skill_vs_persist=1 - mean_squared_error(y, p) / mean_squared_error(y, persist))

rows, preds = [], {}
n_ch, n_ctx = SEQ.shape[1], CTX.shape[1]
for name, mk in [('MLP-v2 traj', lambda: MLPv2(n_ctx)), ('GRU+ctx traj', lambda: GRUCtx(n_ch, n_ctx)), ('TCN+ctx traj', lambda: TCNCtx(n_ch, n_ctx))]:
    torch.manual_seed(SEED)
    t0 = time.time(); m = train(mk())
    P = predict(m, te)
    for h, s in HORIZONS.items():
        y = TRAJ[te, s - 1] + po[te]
        rows.append(dict(horizon_min=h, model=name, **met(y, P[:, s - 1], po[te]), train_min=(time.time() - t0) / 60))
        preds[f'{h}|{name}'] = np.clip(P[:, s - 1], 0, 6)
    preds[f'traj|{name}'] = P.astype('float32')
    print(pd.DataFrame(rows).round(4).tail(3))
res = pd.DataFrame(rows); res.to_csv('resultados_colab/metricas_gru_tcn.csv', index=False)
np.savez_compressed('resultados_colab/predicciones_gru_tcn.npz', **preds)
res.round(4)

In [ ]:
!zip -qr resultados_colab.zip resultados_colab
files.download('resultados_colab.zip')